# 融合模式：什么值得融合、怎么组织


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 融合的收益来源

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">收益</th><th style="text-align:left;">机理</th></tr>
<tr><td>省 GM 往返</td><td>中间结果走片上通路（L0C→UB），不写回 GM 再读出</td></tr>
<tr><td>省 Kernel 启动与同步</td><td>多个算子一个 Kernel，Host 侧一次下发</td></tr>
<tr><td>流水重叠</td><td>Cube 与 Vector 的不同任务重叠执行；两侧利用率仍取决于各阶段耗时与依赖</td></tr>
<tr><td>数据局部性</td><td>Tile 级中间结果留在片上，减少重复搬运</td></tr>
</table>


## 2. 融合收益的量化直觉（一个算例）

以 `Softmax(A×B)` 为例（[1024,128] × [128,1024] FP16，S/out 均为 FP32）：A/B 共 0.5 MiB，S 为 4 MiB，out 也为 4 MiB（1 MiB=2²⁰ B）。

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">方案</th><th style="text-align:left;">GM 流量估算</th><th style="text-align:left;">启动</th></tr>
<tr><td>分离两算子，假定 S 只读一次</td><td>读 A/B 0.5 + 写 S 4 + 读 S 4 + 写 out 4 = 12.5 MiB</td><td>2 次</td></tr>
<tr><td>融合 workspace，同样只读一次 S</td><td>仍为 12.5 MiB；下一节三遍读取 S 的实现为 20.5 MiB</td><td>1 次</td></tr>
<tr><td>理想片上保留全部中间结果</td><td>仅读 A/B 及写 out，共 4.5 MiB</td><td>1 次</td></tr>
</table>

以上忽略分块导致的输入重读等开销；最后一行是流量下界，能否达到取决于片上容量与算法。节省时间可粗估为“减少的字节数 / 有效带宽 + 减少的启动开销”，还需考虑计算与搬运重叠，不能把字节数与时间直接相加。


## 3. 三类典型融合形态

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">形态</th><th style="text-align:left;">结构</th><th style="text-align:left;">例子</th></tr>
<tr><td><b>后处理型</b></td><td>Cube 产出 → Vector 逐元素/归约</td><td>GEMM+Bias+GELU、Matmul+Softmax</td></tr>
<tr><td><b>前置处理型</b></td><td>Vector 预处理 → Cube 消费</td><td>量化/布局准备 + matmul</td></tr>
<tr><td><b>交替流水型</b></td><td>C→V→C→V 多段链</td><td>FlashAttention（QK^T → softmax → ×V）</td></tr>
</table>

选择判断：中间数据被**消费一次** → 片上通路后处理型最简；被**多段反复消费** → workspace 或交替流水；数据量超过片上容量 → 分块 + workspace。


## 4. 官方示例走读：Matmul+Softmax

计算 `Softmax(A × B)`：矩阵乘在 Cube 侧，按行 Softmax 在 Vector 侧，中间矩阵 S 走 **GM workspace**。设计要点（对照前两节概念）：

1. **任务映射**：逻辑 Block 沿 M 轴以 64 行为单位按 `get_block_num()` 跨步分配（Cube 与 Vector 两侧同一映射）；
2. **Cube 侧**：[64,64] 输出 Tile，沿 N 轴逐块 matmul 并 store 到 workspace；写完本组全部区域后 `set_cross_core(pipe=FIX, event_id=0)`；
3. **Vector 侧**：`wait_cross_core(pipe=MTE2)` 后进入；本组 64 行由两个 AIV 各处理 32 行（`get_subblock_idx()` 切分）；
4. **三遍法 Softmax**：完整 N 轴放不进 UB——第一遍沿 N 块合并行最大值，第二遍累加指数和，第三遍归一化写回；
5. **尾块**：M/N 尾块通过 `valid_shape` 传递到 Cube 和 Vector 两侧。

核心代码骨架（节选，完整版见官方文档）：

```python
@pl.jit(auto_mutex=True)
def matmul_softmax_kernel(a, b, out, workspace):
    M, N = a.shape[0], b.shape[1]
    core_id = pl.get_block_idx() // pl.get_subblock_num()
    num_cores = pl.get_block_num()
    # ... Cube/Vector TileGroup 创建（略）...

    with pl.section_cube():
        for mi in pl.range(core_id, M_TILES, num_cores):
            for nj in pl.range(0, N_TILES, 1):
                # set_validshape → load a/b → move → matmul → store workspace
                ...
        pl.system.set_cross_core(pipe=pl.PipeType.FIX, event_id=0)

    with pl.section_vector():
        sub_id = pl.get_subblock_idx()
        pl.system.wait_cross_core(pipe=pl.PipeType.MTE2, event_id=0)
        for mi in pl.range(core_id, M_TILES, num_cores):
            m_off = mi * TILE_M + sub_id * VEC_ROWS   # AIV0/AIV1 各 32 行
            # 三遍：合并行最大值 → 累加指数和 → 归一化写回
            ...
```

【实现骨架，非完整可运行代码：省略参数类型、Tile 分配及 Softmax 运算，需补齐后在 950 验证】

**走读收获**：注意这个示例用 workspace 而非片上通路（S 需要按完整行三遍读取）、注意事件只在本组发一次（组内 Cube/Vector 不按 N 块重叠——这是"正确但未流水化"的版本；流水化改造见 05.03 与第 8 章）。


## 5. 进阶形态预告

官方文档在 Matmul+Softmax 之后提供了使用 **VF 函数与预取流水**的 FlashAttention 进阶示例——它就是第 10 章综合实战的直接底座。该示例还涉及 VF 归约与预取细节，需结合第 8、10 章继续阅读；它不能替代本课程尚待补齐的完整实现。


### 为什么先保留 GM workspace 版本？

每个任务中间结果为 128×128×4=64 KiB，先写 workspace，再由两个 AIV 各读取 32 KiB。单槽预留 64 KiB，两槽预留 128 KiB；4 个任务的中间写读合计 512 KiB。更换槽深度改变容量和可重叠窗口，并不改变这份逻辑流量。

这个版本把读写起点、事件和数值都显式暴露出来，适合先验收。后续改为 L0C→UB 片上交接时，才有稳定的功能基线可对照；不能把“融合到同一 Kernel”直接等同于“消除了所有 GM 中间流量”。完整例子的 GELU 使用稳定的 exp(-2*abs(u))，避免负 u 导致指数溢出，输入仍限定为不会让 x³ 溢出的有限范围。


## 6. 课后练习

1. （选择）GEMM+Bias+GELU 的融合形态属于：
   A. 后处理型<br/>B. 前置处理型<br/>C. 交替流水型
2. （填空）Matmul+Softmax 示例中，两个 AIV 的行切分靠 ______ 区分；Cube 写完 workspace 后从 ______ 流水发事件。
3. （实践）把 Matmul+Softmax 的"S 走 workspace"改为"L0C move 到 UB 的片上通路"，需要满足什么条件？（从 shape/layout/容量三方面分析）

In [ ]:
!cat ./answer/05.04_answer.txt

## 7. 本节小结

- 收益四来源：省 GM 往返、省启动、流水重叠、数据局部性；
- 三形态：**后处理 / 前置 / 交替流水**，按中间数据消费方式选；
- Matmul+Softmax 是"正确性优先"的融合范本：任务映射一致、三遍法、组级事件；
- 正确版之后，下一步是流水化（05.05 实践）。